# TrialScreen: full pipeline (Google Colab or Kaggle)

**Colab:** Runtime → Change runtime type → **T4 GPU**. Internet is on by default.
**Kaggle:** Settings → Accelerator **GPU T4**, **Internet On** (needs a phone-verified account).

Then Runtime → Run all. Roughly 60 to 90 minutes on a T4.

In [ ]:
# 0. Detect the platform and check GPU + internet
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/content" if IN_COLAB else ("/kaggle/working" if IN_KAGGLE else os.getcwd())
print("platform:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "other", "| work dir:", WORK)
net = subprocess.run(["curl", "-sI", "--max-time", "10", "https://github.com"], capture_output=True, text=True).stdout.split("\n")[0]
assert net.startswith("HTTP"), "No internet. Kaggle: Settings > Internet On (phone verification needed). Colab: reconnect the runtime."
import torch
assert torch.cuda.is_available(), "No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4."
print("internet OK |", torch.cuda.get_device_name(0))

In [ ]:
# 1. Get the code
REPO = "https://github.com/eman-huda/trialscreen.git"
ROOT = f"{WORK}/trialscreen"
if not os.path.exists(f"{ROOT}/pyproject.toml"):
    subprocess.run(["rm", "-rf", ROOT])
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
os.chdir(ROOT)
print(os.getcwd()); print(sorted(os.listdir(".")))

In [ ]:
# 2. Install (versions the code was tested with)
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0" scikit-learn matplotlib pytest
!pip install -q -e .
# Kaggle preinstalls torchao 0.10, which recent peft rejects at LoRA creation; we do not use torchao
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data: fixed 37/8/8 patient split (seed 42), leakage checks, unit tests

In [ ]:
!python scripts/01_prepare_data.py
!python -m pytest -q tests | tail -3

## Task 1: LoRA fine-tuning (train split; best epoch by validation macro-F1)

In [ ]:
!python scripts/02_train_lora.py

## Before/after evaluation
Validation first. The test split is evaluated **once**, after everything is frozen.

In [ ]:
!python scripts/03_evaluate.py --split validation

In [ ]:
!python scripts/03_evaluate.py --split test --final

## Task 3: KV-cache benchmark

In [ ]:
!python scripts/04_bench_kv.py

## Task 4: tiled online-softmax attention

In [ ]:
!python scripts/05_attention.py

## Task 5: speculative decoding

In [ ]:
!python scripts/06_bench_speculative.py

## Report tables, figures and error-analysis worksheet

In [ ]:
!python scripts/07_make_report_assets.py
!ls report/figures report/tables

## Publish the adapter to Hugging Face (needed by the app)
Add a secret named `HF_TOKEN` holding a Hugging Face **write** token:
Colab: key icon in the left sidebar → add `HF_TOKEN`, enable notebook access. Kaggle: Add-ons → Secrets.

In [ ]:
HF_REPO = "eman-huda/qwen2.5-1.5b-trialscreen-lora"   # change if your Hugging Face username differs
if IN_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python scripts/08_push_adapter.py --repo {HF_REPO}

## Download everything

In [ ]:
out = f"{WORK}/trialscreen_outputs.zip"
!cd {ROOT} && zip -qr {out} results report checkpoints/qwen2.5-1.5b-trialscreen-lora/best_adapter data/splits
!ls -lh {out}
if IN_COLAB:
    from google.colab import files
    files.download(out)
else:
    print("Kaggle: download it from the Output panel on the right.")